# **Sesión 4 - Análisis y visualización de información química**

# Target: Amyloid-beta precursor protein

Although clinical trials targeting Aβ have yielded disappointing results, its central role in plaque formation and disease pathology makes it a biologically validated and structurally well-studied target. Virtual screening provides an opportunity to identify novel small molecules that may interfere with fundamental pathways in ways not previously explored.


* Zhang, Y., Chen, H., Li, R. et al. Amyloid β-based therapy for Alzheimer’s disease: challenges, successes and future. Sig Transduct Target Ther 8, 248 (2023). https://doi.org/10.1038/s41392-023-01484-7

* Karahan H, Hartigan K, Al-Amin MM, et al. Deletion of neuronal Idol ameliorates Alzheimer's disease–related pathologies via APOE receptors. Alzheimer's Dement. 2025; 21:e70949. https://doi.org/10.1002/alz.70949


Amyloid-beta precursor protein (APP) / P05067 · A4_HUMAN

* https://www.uniprot.org/uniprotkb/P05067/entry



In [7]:
# Importing necessary libraries

import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import urllib.request

import pip
! pip install rdkit
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import Draw

! pip install chembl_webresource_client
from chembl_webresource_client.new_client import new_client as search
from chembl_webresource_client.utils import utils

testdf = pd.read

In [ ]:
# https://pubchem.ncbi.nlm.nih.gov/rest/pug/<input specification>/<operation specification>/[<output specification>][?<operation_options>]


**Bioactivities from Protein - Pubchem**

This operation returns the concise bioactivity data for a given protein. Valid output formats are XML, JSON(P), ASNT/B, and CSV.

https://pubchem.ncbi.nlm.nih.gov/rest/pug/protein/accession/Q01279/concise/JSON

For some proteins with a large amount of data, the operation may time out. In such cases, one can first get the list of AIDs tested against the given protein, e.g.:

https://pubchem.ncbi.nlm.nih.gov/rest/pug/protein/accession/Q01279/aids/TXT

Then aggregate the concise bioactivity data from each AID:

https://pubchem.ncbi.nlm.nih.gov/rest/pug/assay/aid/66438/concise/JSON

In [58]:
# Bioactivities from Protein

# URL = prolog + / + input + / + operation + / + output + options

# Uniprot ID of the target protein: P05067

target = 'P05067'

prolog = 'https://pubchem.ncbi.nlm.nih.gov/rest/pug'


# 🟢 Input
# <domain> = substance | compound | assay | gene | protein | pathway | taxonomy | cell | <other inputs>
# protein domain <namespace> = accession | gi | synonym
input = f'protein/accession/{target}'

# 🟡 Operation
# protein domain <operation specification> = summary | aids | concise | pwaccs
operation = 'concise'

# 🔵 output
# <output specification> = XML | ASNT | ASNB | JSON | JSONP [ ?callback=<callback name> ] | SDF | CSV | PNG | TXT
output = 'CSV'

# 🔴 Options
options = '' # '' Si no se requiere nada en esoecific

# URL construction
url = f'{prolog}/{input}/{operation}/{output}{options}'
print(f'🔗 La URL de búsqueda en Pubchem es: {url}')

🔗 La URL de búsqueda en Pubchem es: https://pubchem.ncbi.nlm.nih.gov/rest/pug/protein/accession/P05067/concise/CSV


In [73]:
# Creation of a dataset from search results

app_df_pubchem = pd.read_csv(url, low_memory=False)
print(f'📊 El número de compuestos con información de bioactividad para la proteína APP es: {len(app_df_pubchem)}\n')
app_df_pubchem.describe()

📊 El número de compuestos con información de bioactividad para la proteína APP es: 418493



,AID,SID,CID,Target GeneID,Activity Value [uM],PubMed ID
count,4.184930e+05,4.184930e+05,4.184440e+05,418154.0,2920.000000,8.825000e+03
mean,1.346027e+06,4.206476e+07,1.496779e+07,351.0,33.758569,2.866043e+07
std,8.093859e+04,5.444924e+07,2.792483e+07,0.0,535.963093,5.600739e+06
min,3.175700e+04,8.421210e+05,6.000000e+00,351.0,0.000070,1.088236e+07
25%,1.347071e+06,1.750599e+07,2.108374e+06,351.0,0.022000,2.523817e+07
50%,1.347071e+06,2.481365e+07,4.290750e+06,351.0,0.679000,2.845814e+07
75%,1.347071e+06,4.973265e+07,1.603086e+07,351.0,10.000000,3.339562e+07
max,2.143379e+06,5.200681e+08,1.776614e+08,351.0,25000.000000,3.925443e+07


In [74]:
app_df_pubchem.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418493 entries, 0 to 418492
Data columns (total 11 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   AID                  418493 non-null  int64  
 1   SID                  418493 non-null  int64  
 2   CID                  418444 non-null  float64
 3   Activity Outcome     418493 non-null  object 
 4   Target GeneID        418154 non-null  float64
 5   Activity Name        4410 non-null    object 
 6   Activity Qualifier   2158 non-null    object 
 7   Activity Value [uM]  2920 non-null    float64
 8   Assay Name           418493 non-null  object 
 9   Assay Type           418493 non-null  object 
 10  PubMed ID            8825 non-null    float64
dtypes: float64(4), int64(2), object(5)
memory usage: 35.1+ MB


In [75]:
# Preserving only molecules that contain activity data (IC50, Ki, EC50, etc.)

app_df_pubchem = app_df_pubchem[app_df_pubchem['Activity Value [uM]'].notna()]
print(f'🔶 El número de compuestos con datos de bioactividad para la proteína APP es: {len(app_df_pubchem)}\n')

app_df_pubchem['Activity Name'].value_counts()

🔶 El número de compuestos con datos de bioactividad para la proteína APP es: 2920



Activity Name
IC50        1913
Ki           576
EC50         214
Kd           133
Potency       66
DC50          11
Activity       3
Bmax           3
INH            1
Name: count, dtype: int64

In [76]:
# Presenving only molecules with IC50 values

app_df_pubchem = app_df_pubchem[app_df_pubchem['Activity Name'] == 'IC50']
print(f'🔷 El número de compuestos con datos de IC50 para la proteína APP es: {len(app_df_pubchem)}\n')

app_df_pubchem.sample(5)

🔷 El número de compuestos con datos de IC50 para la proteína APP es: 1913



,AID,SID,CID,Activity Outcome,Target GeneID,Activity Name,Activity Qualifier,Activity Value [uM],Assay Name,Assay Type,PubMed ID
349304,1700052,136939963,969516.0,Unspecified,351.0,IC50,=,16.2500,Inhibition of amyloid beta (1 to 42) (unknown ...,Confirmatory,33137375.0
353611,2060863,513522260,153606637.0,Active,351.0,IC50,NaN,0.0413,Cellular gamma-Secretase Assay from US Patent ...,Confirmatory,NaN
397817,1973788,504826079,172461377.0,Unspecified,351.0,IC50,=,49.3700,Inhibition of amyloid beta (1 to 42) (unknown ...,Confirmatory,36682173.0
418398,1798079,249813236,67979599.0,Active,351.0,IC50,NaN,0.0170,sAPPbeta Release Assay from US Patent US102319...,Confirmatory,NaN
9971,1234678,318377702,122181797.0,Active,351.0,IC50,=,0.8000,Inhibition of Amyloid beta (1 to 42) (unknown ...,Confirmatory,26078011.0


In [77]:
app_df_pubchem['CID'].value_counts()

CID
969516.0       34
445154.0       17
5281672.0       7
57385371.0      6
57404290.0      6
               ..
66591.0         1
5281292.0       1
60172115.0      1
14214498.0      1
154938049.0     1
Name: count, Length: 1475, dtype: int64

In [80]:
# Dropping rows with missing CID values and duplicates
app_df_pubchem = app_df_pubchem.dropna(subset=['CID'])
app_df_pubchem = app_df_pubchem.drop_duplicates(subset=['CID'])
print(f'✅ El número de compuestos únicos con CID válido es: {len(app_df_pubchem)}')

# Converting the CID column to string type for better handling
app_df_pubchem['CID'] = app_df_pubchem['CID'].astype(int)
app_df_pubchem['CID'] = app_df_pubchem['CID'].astype('string')

✅ El número de compuestos únicos con CID válido es: 1475


In [ ]:
# Paso adicional de ajuste por output = 'XML'

cids = []

for i in app_df_pubchem['CID'].sample(50):
    cids.append(i)

print(f'✅ La lista de CIDs limpia es: {cids}')

# Creación del string con los 6 compuestos

cids_string = ','.join(cids)
print(f'🔶 El string continuo de los 6 compuestos es: {cids_string}')

✅ La lista de CIDs limpia es: ['137660068', '172462631', '137654357', '11645550', '10367765', '71288634', '72705388', '136638362', '137638769', '127048771', '145990938', '154938133', '24786216', '71811225', '11565846', '71529373', '135582042', '5281855', '135693471', '154938114', '71274019', '72205405', '24785163', '72714659', '58512491', '166626588', '598987', '70690403', '44425723', '135995958', '72705387', '71764227', '57385625', '162664081', '57386414', '53494589', '54579970', '71811226', '72705233', '5281610', '136505159', '127037724', '57387885', '162677045', '145992158', '73357094', '72205409', '155451940', '137637059', '71273880']
🔶 El string continuo de los 6 compuestos es: 154938113,24752854,67979574,145952951,136036825,122238739,71288697,164617606,24786216,57384875,72205034,10311969,24896605,164624719,155544377,44555289,127037733,57385877,162784594,70690404,164608954,101898808,136276907,44425691,67979538,166633393,52108098,57578199,88982396,72714842,44435395,72205216,5734278

In [85]:
# URL = prolog + / + input + / + operation + / + output + options
cids = cids_string

prolog = 'https://pubchem.ncbi.nlm.nih.gov/rest/pug'
input = f"compound/cid/{cids}"
operation = "property/SMILES"
output = "txt"
options = '' # '' Si no se requiere nada en especifico

# Construcción de la URL
url = f'{prolog}/{input}/{operation}/{output}{options}' # Se cambia el orden del output
print(f'\n🔗 La URL es: {url}')

# Lista con los smiles de los 6 compuestos
cids20 = requests.get(url)
cids20_list = cids20.text.split()
print(f'\n6️⃣ La lista de los 6 smiles es: {cids20_list}')



🔗 La URL es: https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/137660068,172462631,137654357,11645550,10367765,71288634,72705388,136638362,137638769,127048771,145990938,154938133,24786216,71811225,11565846,71529373,135582042,5281855,135693471,154938114,71274019,72205405,24785163,72714659,58512491,166626588,598987,70690403,44425723,135995958,72705387,71764227,57385625,162664081,57386414,53494589,54579970,71811226,72705233,5281610,136505159,127037724,57387885,162677045,145992158,73357094,72205409,155451940,137637059,71273880/property/SMILES/txt

6️⃣ La lista de los 6 smiles es: ['COC1=C(C=C(C=C1)CCNC2=NC(=O)NC3=CC=CC=C32)OC', 'CN1CCC2=CC3=C(C4=C2[C@H]1CC5=C4C(=CC(=C5)O)OC)OCO3', 'C1=CC=C(C=C1)CCNC2=NC(=O)NC3=CC=CC=C32', 'CC(C)(C1=CC(=C(C=C1)C2=CC=C(C=C2)C(F)(F)F)F)C(=O)O', 'CCN(CC)CCCOC1=CC=C(C=C1)C2=C(C3=CC=CC=C3O2)C(=O)C4=CC=C(C=C4)C', 'CCO[C@@H]([C@H]1C[C@H]([C@H]2[C@H](O1)[C@@H]([C@@]3([C@@]2(CC[C@]45[C@H]3CC[C@@H]6[C@]4(C5)CC[C@@H](C6(C)C)O[C@H]7CN(CCO7)CC8CN(C8)C9COC9)C)C)O)C